# Sigmoid Perceptron — Theory & Math

---

## Table of Contents

1. [What is a Perceptron](#1-what-is-a-perceptron)
2. [Weights and Bias](#2-weights-and-bias)
3. [The Weighted Sum](#3-the-weighted-sum)
4. [Sigmoid Function](#4-sigmoid-function)
5. [Making a Prediction](#5-making-a-prediction)
6. [Measuring Error](#6-measuring-error)
7. [Derivative of Sigmoid](#7-derivative-of-sigmoid)
8. [Gradient Descent](#8-gradient-descent)
9. [Updating Weights and Bias](#9-updating-weights-and-bias)
10. [Training Loop](#10-training-loop)
11. [Feature Scaling](#11-feature-scaling)
12. [Evaluation](#12-evaluation)
13. [Full Example — Student Pass/Fail](#13-full-example--student-passfail)

---

## 1. What is a Perceptron

A perceptron is the smallest possible neural network — just one neuron. It takes a few numbers as input, does some math, and outputs a number between 0 and 1 which you can read as a probability.

```
x1 --(w1)--\
x2 --(w2)---+-- [ weighted sum + bias ] -- [ sigmoid ] --> output
x3 --(w3)--/
```

Think of it like a person making a yes/no decision. They look at a few factors, give each factor some importance, add everything up, and decide. The perceptron does exactly that but with numbers.

---

## 2. Weights and Bias

Every input has a weight. The weight controls how much that input matters to the final output.

- large positive weight → input pushes the output toward 1
- large negative weight → input pushes the output toward 0  
- weight near zero → input barely matters

Weights are initialized randomly from a normal distribution:

$$w \sim \mathcal{N}(0,\ 1)$$

The bias is one extra number that shifts the output regardless of inputs. It gives the model a default starting lean before it looks at any data.

```python
self.weights = np.random.randn(input_size)
self.bias    = np.random.randn()
```

**Why not start with zeros?**

If every weight is zero then:

$$z = x_1 \cdot 0 + x_2 \cdot 0 + \ldots = 0$$

Every input gives the exact same output. The model has no way to tell inputs apart and learns nothing. Random initialization breaks this symmetry.

---

## 3. The Weighted Sum

Multiply each input by its weight, add them all together, then add the bias. The result is called $z$.

$$z = x_1 w_1 + x_2 w_2 + \ldots + x_n w_n + b$$

Using dot product notation (shorter way to write the same thing):

$$z = \mathbf{x} \cdot \mathbf{w} + b$$

Example with two features — study hours and sleep hours:

| Feature | Value $x$ | Weight $w$ | $x \cdot w$ |
|---|---|---|---|
| Study hours | 7 | 1.2 | 8.4 |
| Sleep hours | 8 | 0.6 | 4.8 |
| bias | — | — | −0.5 |
| **z** | | | **12.7** |

```python
z = np.dot(inputs, self.weights) + self.bias
```

`np.dot` handles all the multiplications and additions in one call. The result $z$ can be any number — negative, zero, or huge positive. That is why sigmoid comes next.

---

## 4. Sigmoid Function

Sigmoid takes $z$ and squishes it into a value strictly between 0 and 1.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

where $e = 2.71828\ldots$ (Euler's number)

The shape is an S-curve:

```
1.0 |                   ─────────────
    |                 /
0.5 |. . . . . . . ./. . . . . . . .   <-- 0.5 threshold
    |             /
0.0 |────────────/
    +─────────────────────────────────
        -5    -2    0    2    5     z
```

Some values to get a feel for it:

| $z$ | $\sigma(z)$ |
|---|---|
| −5 | 0.007 |
| −1 | 0.269 |
| 0 | 0.500 |
| +1 | 0.731 |
| +5 | 0.993 |

```python
def sigmoid(self, Z):
    return 1 / (1 + np.exp(-Z))
```

The reason sigmoid is used here instead of just raw $z$ is that $z = 12.7$ means nothing as a probability. After sigmoid, $\sigma(12.7) \approx 0.9999$ — now it reads as 99.99% chance of yes, which actually makes sense.

---

## 5. Making a Prediction

A prediction is just running an input through both steps:

$$\hat{y} = \sigma(\mathbf{x} \cdot \mathbf{w} + b)$$

$\hat{y}$ (pronounced y-hat) is the predicted probability. Close to 1 means the model thinks yes, close to 0 means no.

```python
def predict(self, inputs):
    z = np.dot(inputs, self.weights) + self.bias
    return self.sigmoid(z)
```

---

## 6. Measuring Error

After the model makes a prediction we check how wrong it was:

$$\text{error} = y - \hat{y}$$

where $y$ is the real answer (0 or 1) and $\hat{y}$ is what the model predicted.

| Real $y$ | Predicted $\hat{y}$ | Error | What it means |
|---|---|---|---|
| 1 | 0.3 | +0.7 | predicted too low, push up |
| 0 | 0.8 | −0.8 | predicted too high, push down |
| 1 | 0.95 | +0.05 | almost right, tiny fix |
| 0 | 0.03 | −0.03 | almost right, tiny fix |

Positive error means the prediction was too low. Negative error means too high. The sign tells the model which direction to move.

---

## 7. Derivative of Sigmoid

To update weights correctly, we need to know how steep the sigmoid curve is at the current prediction point. This comes from calculus — specifically the derivative.

The derivative of sigmoid works out to:

$$\frac{d\sigma}{dz} = \sigma(z)\,(1 - \sigma(z)) = \hat{y}\,(1 - \hat{y})$$

This is one of the nicest results in neural network math — the derivative of sigmoid is just sigmoid times one minus sigmoid. No need to compute anything new.

What this number tells you:

| $\hat{y}$ | $\hat{y}(1-\hat{y})$ | Meaning |
|---|---|---|
| 0.50 | 0.250 | maximum steepness, biggest possible update |
| 0.70 | 0.210 | still quite steep |
| 0.90 | 0.090 | getting flatter, smaller update |
| 0.99 | 0.010 | nearly flat, very small update |

The model automatically makes big adjustments when it is uncertain (near 0.5) and small adjustments when it is confident (near 0 or 1). This behavior comes directly from the math, not from any special logic written in code.

---

## 8. Gradient Descent

Gradient descent is how the perceptron gets better over time. The idea is simple — if you can measure which direction makes the error bigger, you can move in the opposite direction to make it smaller.

Imagine standing blindfolded on a hilly landscape. You want to reach the lowest point. You feel the slope under your feet and take a small step downhill. You repeat this until you stop going down.

```
error
  |
  |\
  | \
  |  \          steep here → big step
  |   \_
  |     \__
  |        \__________
  +─────────────────────────> weight value
                   ^
               minimum error
```

The gradient at any point is:

$$g = \text{error} \times \hat{y}\,(1 - \hat{y})$$

This combines how wrong the model is (error) with how steep the sigmoid curve is at that point. Together they give the direction and rough size of the step to take.

---

## 9. Updating Weights and Bias

These update rules come from backpropagation, which applies the chain rule from calculus to figure out exactly how much each weight contributed to the error.

**Weight update:**

$$\Delta w = \alpha \cdot \text{error} \cdot \hat{y}\,(1-\hat{y}) \cdot x$$

$$w_\text{new} = w_\text{old} + \Delta w$$

**Bias update:**

$$\Delta b = \alpha \cdot \text{error} \cdot \hat{y}\,(1-\hat{y})$$

$$b_\text{new} = b_\text{old} + \Delta b$$

$\alpha$ is the **learning rate** — a small number like 0.1 that controls how big each step is.

The bias update has no $x$ at the end because bias is not connected to any specific input. It shifts the output by a flat amount regardless of what the inputs are, so there is nothing to scale by.

**On learning rate:**

| $\alpha$ | what happens |
|---|---|
| too large (e.g. 5.0) | overshoots the minimum, bounces around, never settles |
| just right (e.g. 0.1) | steady descent, converges smoothly |
| too small (e.g. 0.000001) | takes forever to get anywhere |

```python
gradient_weights = error * prediction * (1 - prediction) * input_vector
self.weights += learning_rate * gradient_weights

gradient_bias = error * prediction * (1 - prediction)
self.bias += learning_rate * gradient_bias
```

---

## 10. Training Loop

One **epoch** is one full pass through the entire training dataset. We run multiple epochs because a single pass is not enough — the model needs to see the same data many times before it gets good at it.

Each epoch the model goes through every training example one by one:

```
for each epoch:
    for each training example:
        1. predict          y_hat = sigmoid(x·w + b)
        2. measure error    error = y - y_hat
        3. update weights   w += lr * error * y_hat*(1-y_hat) * x
        4. update bias      b += lr * error * y_hat*(1-y_hat)
    print loss
```

Total weight updates = epochs × training examples. With 100 epochs and 8 students that is 800 updates.

---

## 11. Feature Scaling

If features have very different ranges, the ones with large values will dominate the weight updates and the ones with small values will barely matter. Scaling fixes this by putting every feature on the same scale.

The formula is called z-score normalization:

$$x_\text{scaled} = \frac{x - \mu}{\sigma}$$

where $\mu$ is the mean and $\sigma$ is the standard deviation of that feature across the training set.

After scaling every feature has mean 0 and standard deviation 1. Most values end up between −3 and +3.

```python
mean = X_train.mean(axis=0)
std  = X_train.std(axis=0)

X_train_scaled = (X_train - mean) / std
X_test_scaled  = (X_test  - mean) / std
```

Note that the test set is scaled using the mean and std from the training set, not its own. If you compute stats from the test set you are leaking information about it into your preprocessing, which makes your results dishonest.

---

## 12. Evaluation

After training we measure accuracy on data the model has never seen:

$$\text{accuracy} = \frac{\text{correct predictions}}{\text{total predictions}}$$

To go from a probability to a class label we use 0.5 as the cutoff:

$$\text{predicted class} = \begin{cases} 1 & \text{if } \hat{y} \geq 0.5 \\ 0 & \text{if } \hat{y} < 0.5 \end{cases}$$

```python
def evaluate(self, inputs, targets):
    correct = 0
    for input_vector, target in zip(inputs, targets):
        prediction = self.predict(input_vector)
        predicted_class = 1 if prediction >= 0.5 else 0
        if predicted_class == target:
            correct += 1
    return correct / len(inputs)
```

---

## 13. Full Example — Student Pass/Fail

Training a perceptron to predict whether a student passes based on study hours and sleep hours. No sklearn, no pandas — only numpy.

**Dataset — 10 students:**

| Student | Study hrs | Sleep hrs | Result |
|---|---|---|---|
| 1 | 9 | 8 | Pass |
| 2 | 8 | 7 | Pass |
| 3 | 7 | 8 | Pass |
| 4 | 6 | 6 | Pass |
| 5 | 5 | 7 | Pass |
| 6 | 4 | 5 | Fail |
| 7 | 3 | 4 | Fail |
| 8 | 2 | 6 | Fail |
| 9 | 1 | 3 | Fail |
| 10 | 2 | 4 | Fail |


## The 3 Core Equations

Everything the perceptron does comes down to three equations. Forward pass to predict, error to measure, backward pass to learn.

**Forward pass:**
$$\hat{y} = \frac{1}{1 + e^{-(\mathbf{x}\cdot\mathbf{w}+b)}}$$

**Error:**
$$\text{error} = y - \hat{y}$$

**Backward pass:**
$$w \mathrel{+}= \alpha \cdot \text{error} \cdot \hat{y}(1-\hat{y}) \cdot x$$
$$b \mathrel{+}= \alpha \cdot \text{error} \cdot \hat{y}(1-\hat{y})$$

Every neural network in existence — regardless of size — is built on these same ideas. The only difference is more layers and more data.